# 🎒 Python Knapsack DP — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> The knapsack problem is like packing a bag with a weight limit.
> Each item has a weight and a value. You want maximum value without exceeding the limit.
> The 0/1 variant: each item can be taken or left — you decide once.
> Subset Sum variant: can you hit exactly a target weight?
> Unbounded: you can take an item as many times as you like.
> DP builds up the answer by considering items one at a time.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Knapsack DP? The Visual Model](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [The Core API — All Variants](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Partition Equal Subset Sum (LC 416)](#5) |
| 6 | [Pattern 2: Target Sum / Count Subsets (LC 494)](#6) |
| 7 | [Pattern 3: Ones and Zeroes / 2D Knapsack (LC 474)](#7) |
| 8 | [Pattern 4: Coin Change 2 — Count Ways (LC 518)](#8) |
| 9 | [The Knapsack Decision Map](#9) |
| 10 | [Interview Cheat Sheet](#10) |

<a id='1'></a>
## 1. What Is Knapsack DP? The Visual Model

```
               KNAPSACK — THE PACKING PROBLEM

  0/1 KNAPSACK — items: [{w=1,v=1},{w=2,v=6},{w=3,v=10}], capacity=5

  dp[cap] = max value achievable with capacity=cap

  After each item (process one item at a time, RIGHT TO LEFT to avoid reuse):

  cap:    0   1   2   3   4   5
  init:   0   0   0   0   0   0
  item1(w=1,v=1): 0   1   1   1   1   1
  item2(w=2,v=6): 0   1   6   7   7   7
  item3(w=3,v=10):0   1   6   10  11  16

  answer = dp[5] = 16 (take item2 + item3: v=6+10, w=2+3=5)

  KEY RULE FOR 0/1 KNAPSACK:
  Process capacity RIGHT TO LEFT when using 1D array.
  This ensures each item is counted at most ONCE.
  (Going left→right would allow the same item to be counted multiple times)

  UNBOUNDED KNAPSACK (reuse items) → go LEFT TO RIGHT
  0/1 KNAPSACK (each item once) → go RIGHT TO LEFT
  SUBSET SUM → dp[target] is bool: True if reachable
  COUNT WAYS → dp[target] accumulates count instead of max
```

<a id='2'></a>
## 2. Creating / Setup

In [ ]:
# BASIC KNAPSACK TEMPLATES

# 0/1 KNAPSACK — max value, each item at most once
def knapsack_01(weights, values, capacity):
    dp = [0] * (capacity + 1)
    for w, v in zip(weights, values):
        for cap in range(capacity, w - 1, -1):  # RIGHT TO LEFT — each item used once
            dp[cap] = max(dp[cap], dp[cap - w] + v)
    return dp[capacity]

# UNBOUNDED KNAPSACK — max value, items reusable
def knapsack_unbounded(weights, values, capacity):
    dp = [0] * (capacity + 1)
    for cap in range(1, capacity + 1):          # LEFT TO RIGHT — items reusable
        for w, v in zip(weights, values):
            if w <= cap:
                dp[cap] = max(dp[cap], dp[cap - w] + v)
    return dp[capacity]

# SUBSET SUM — can we reach exactly the target?
def subset_sum(nums, target):
    dp = [False] * (target + 1)
    dp[0] = True                               # empty subset sums to 0
    for num in nums:
        for cap in range(target, num - 1, -1): # RIGHT TO LEFT
            dp[cap] = dp[cap] or dp[cap - num]
    return dp[target]

# Demo
weights = [1, 2, 3]
values  = [1, 6, 10]
capacity = 5
print("0/1 knapsack:",       knapsack_01(weights, values, capacity))         # 16
print("unbounded knapsack:", knapsack_unbounded(weights, values, capacity))  # 18 (3×item2)
print("subset sum [1,2,3] target=5:", subset_sum([1,2,3], 5))  # True (2+3)
print("subset sum [1,2,3] target=7:", subset_sum([1,2,3], 7))  # False
print("Knapsack templates loaded.")

<a id='3'></a>
## 3. The Core API — All Variants

```
VARIANT               INIT         LOOP ORDER    RECURRENCE
───────────────────────────────────────────────────────────────────────
0/1 max value         dp[0]=0      R→L per item  dp[c]=max(dp[c],dp[c-w]+v)
0/1 subset sum        dp[0]=True   R→L per item  dp[c]=dp[c] or dp[c-w]
0/1 count ways        dp[0]=1      R→L per item  dp[c]+=dp[c-w]
Unbounded max value   dp[0]=0      L→R per cap   dp[c]=max(dp[c],dp[c-w]+v)
Unbounded count ways  dp[0]=1      L→R per cap   dp[c]+=dp[c-w]
2D knapsack           dp[0][0]=v   R→L both dims dp[i][j]=max(...)

CRITICAL RULE:
  0/1 (each item ONCE): iterate capacity RIGHT→LEFT (large to small)
      → when processing dp[c], dp[c-w] still reflects the state BEFORE this item
  UNBOUNDED (items REUSABLE): iterate capacity LEFT→RIGHT (small to large)
      → when processing dp[c], dp[c-w] may already include this item = reuse!

THINGS YOU DO NOT DO:
❌  Use left→right for 0/1 knapsack — item gets counted multiple times
❌  Use right→left for unbounded — item can only be used once, defeating the purpose
❌  Initialize dp[0]=0 for count-ways problems (should be dp[0]=1 as the base)
❌  Forget to handle the case where num > target in the inner loop
```

In [ ]:
# DEMO: why loop direction matters

# LEFT→RIGHT: same item CAN be reused
nums = [2, 3]
target = 6

dp_left_right = [False] * (target + 1)
dp_left_right[0] = True
for num in nums:
    for c in range(num, target + 1):        # left to right — reuse allowed
        dp_left_right[c] = dp_left_right[c] or dp_left_right[c - num]
print("left→right (unbounded), target=6:", dp_left_right[6])  # True (2+2+2)

dp_right_left = [False] * (target + 1)
dp_right_left[0] = True
for num in nums:
    for c in range(target, num - 1, -1):   # right to left — no reuse (0/1)
        dp_right_left[c] = dp_right_left[c] or dp_right_left[c - num]
print("right→left (0/1), target=6:     ", dp_right_left[6])   # False (no way: 2+3=5, 2+2=4, etc.)
print()
print("Direction matters: L→R=unbounded, R→L=0/1 (each item once)")

<a id='4'></a>
## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                   VARIANT       LOOP ORDER
──────────────────────────────────────────────────────────────────────
"can split into two equal subsets"      0/1 subset     R→L
"count ways to assign +/- to nums"      0/1 count      R→L
"max # strings using at most m 0s n 1s" 2D knapsack    R→L both dims
"count ways to make change"             unbounded cnt  L→R
"min/max value, items reusable"         unbounded max  L→R
"can we form exactly a target weight"   0/1 bool       R→L
"partition into subsets with equal sum" 0/1 subset     R→L
"bounded items (each item 0 or 1)"      0/1            R→L
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Partition Equal Subset Sum — LC 416

---

```
PROBLEM:
  Given a non-empty array, determine if it can be partitioned into two subsets
  with equal sum.

TRICK:
  Equal partition → each subset has sum = total/2.
  If total is odd → impossible immediately.
  Reduce to subset sum: can we find a subset summing to total//2?
  0/1 knapsack (each number used at most once), target = total//2.

SLOW MOTION TRACE on nums=[1,5,11,5]:

  total=22, target=11
  dp = [T,F,F,F,F,F,F,F,F,F,F,F]  (indices 0..11)

  process 1: R→L from 11 to 1
    dp[1] = dp[1] or dp[0] = T  → dp=[T,T,F,...]

  process 5: R→L from 11 to 5
    dp[6]=T (5+1), dp[5]=T (5+0) → dp=[T,T,F,F,F,T,T,F,...]

  process 11: R→L from 11 to 11
    dp[11]=T (11+0) → immediately True

  return dp[11] = True

KEY INSIGHT:
  Two equal halves ↔ one half sums to total//2 ↔ subset sum problem.
  Early exit: if dp[target] becomes True during iteration, we're done.

TIME:  O(n * target) = O(n * sum/2)
SPACE: O(target)
```

In [ ]:
def can_partition(nums):
    """
    LC 416 — Partition Equal Subset Sum
    Approach: 0/1 subset sum DP — find if any subset sums to total//2.
    Args:
        nums (List[int]): positive integers.
    Returns:
        bool: True if array can be partitioned into two equal-sum subsets.
    Time:  O(n * S/2) — S = sum of array, n = len(nums)
    Space: O(S/2) — 1D dp array of size target+1
    """
    total = sum(nums)
    if total % 2 != 0:
        return False          # odd total → can't split equally
    target = total // 2

    dp = [False] * (target + 1)
    dp[0] = True              # empty subset sums to 0

    for num in nums:
        if num > target:
            return False      # single number exceeds half-sum → impossible
        for cap in range(target, num - 1, -1):  # RIGHT TO LEFT: each num used once
            dp[cap] = dp[cap] or dp[cap - num]  # can we reach cap using this num?
        if dp[target]:
            return True       # early exit — found a valid partition

    return dp[target]

# Slow motion on [1,5,11,5], target=11:
# dp=[T,F,F,F,F,F,F,F,F,F,F,F]
# num=1 R→L: dp[1]=dp[0]=T → dp=[T,T,F,...]
# num=5 R→L: dp[6]=dp[1]=T, dp[5]=dp[0]=T → dp=[T,T,F,F,F,T,T,F,...]
# num=11: dp[11]=dp[0]=T → return True immediately

def test_harness(fn):
    tests = [
        ([1,5,11,5], True),
        ([1,2,3,5], False),
        ([1,1], True),
        ([1], False),
        ([2,2,3,5], False),     # total=12, target=6: no subset sums to 6
        ([3,3,3,4,5], True),    # total=18, target=9: 3+3+3=9 and 4+5=9
        ([100,100], True),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(inputs[0])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | nums={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(can_partition)
print("can_partition defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Target Sum / Count Subsets — LC 494

---

```
PROBLEM:
  Given nums and a target, assign '+' or '-' to each number.
  Count the number of ways to reach the target sum.

TRICK:
  Let P = set of numbers with '+', N = set with '-'.
  P - N = target, P + N = total.
  → P = (total + target) / 2.
  Reduce to: count subsets with sum = (total + target) / 2.
  0/1 count-ways DP (R→L), dp[0]=1 as base.

SLOW MOTION TRACE on nums=[1,1,1,1,1], target=3:

  total=5, P=(5+3)/2=4
  dp = [1,0,0,0,0]  (only way to make 0 is empty set)

  process 1 (5 times):
  after 1st 1: dp=[1,1,0,0,0]  (pick the 1: make 1)
  after 2nd 1: dp=[1,2,1,0,0]  (two ways to make 1: pick either 1)
  after 3rd 1: dp=[1,3,3,1,0]
  after 4th 1: dp=[1,4,6,4,1]
  after 5th 1: dp=[1,5,10,10,5]

  return dp[4] = 5  (5 ways to assign +/-)

KEY INSIGHT:
  Assign +/- → find subset summing to a derived target.
  Count-ways 0/1 knapsack: dp[c] += dp[c-num] (R→L).

TIME:  O(n * P) where P=(total+target)/2
SPACE: O(P)
```

In [ ]:
def find_target_sum_ways(nums, target):
    """
    LC 494 — Target Sum
    Approach: Reduce to count-ways 0/1 knapsack: count subsets summing to (total+target)//2.
    Args:
        nums (List[int]): non-negative integers.
        target (int): desired sum after assigning + or - to each number.
    Returns:
        int: number of ways to reach the target.
    Time:  O(n * P) — P = (total+target)/2
    Space: O(P) — 1D dp array
    """
    total = sum(nums)
    # check feasibility: target must be reachable
    if abs(target) > total or (total + target) % 2 != 0:
        return 0
    subset_sum_target = (total + target) // 2   # we want subsets of this sum

    dp = [0] * (subset_sum_target + 1)
    dp[0] = 1     # 1 way to achieve sum=0: pick nothing

    for num in nums:
        for cap in range(subset_sum_target, num - 1, -1):  # R→L: each num used once
            dp[cap] += dp[cap - num]   # add the ways to make (cap-num) before this item

    return dp[subset_sum_target]

# Slow motion on [1,1,1,1,1], target=3:
# total=5, P=(5+3)//2=4
# dp=[1,0,0,0,0]
# after 5 passes of num=1: dp=[1,5,10,10,5]
# return dp[4]=5

def test_harness(fn):
    tests = [
        ([1,1,1,1,1], 3, 5),
        ([1], 1, 1),
        ([1], 2, 0),       # impossible
        ([0,0,0,0,0,0,0,0,1], 1, 256),  # 8 zeros each contribute 2 options → 2^8=256
        ([1,2,3,4,5], 3, 3),
    ]
    passed = 0
    for *inputs, expected in tests:
        nums, tgt = inputs
        got = fn(nums, tgt)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | nums={nums} target={tgt} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(find_target_sum_ways)
print("find_target_sum_ways defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Ones and Zeroes / 2D Knapsack — LC 474

---

```
PROBLEM:
  Given strs (binary strings) and limits m (max 0s) and n (max 1s),
  find the size of the largest subset of strs with at most m 0s and n 1s.

TRICK:
  2D knapsack: two capacity dimensions (count of 0s, count of 1s).
  dp[i][j] = max strings we can include using at most i zeros and j ones.
  For each string, count its zeros (z) and ones (o).
  For each (i,j) from (m,n) down to (z,o): dp[i][j] = max(dp[i][j], dp[i-z][j-o]+1)
  Iterate both capacity dimensions RIGHT TO LEFT (0/1 — each string used once).

SLOW MOTION TRACE on strs=["10","0001","111001","1","0"], m=5, n=3:

  string "10": z=1, o=1
    for i=5..1, j=3..1: dp[i][j] = max(dp[i][j], dp[i-1][j-1]+1)
    dp[1][1] = max(0, dp[0][0]+1) = 1

  string "0001": z=3, o=1
    dp[4][2] = max(dp[4][2], dp[1][1]+1) = max(0,2) = 2

  ... continue for all strings
  answer = dp[m][n] = 4

KEY INSIGHT:
  Two resource constraints = two capacity dimensions.
  Same 0/1 knapsack logic, just iterate BOTH dimensions right→left.

TIME:  O(len(strs) * m * n * L) where L = avg string length for counting
SPACE: O(m * n) — 2D dp array
```

In [ ]:
def find_max_form(strs, m, n):
    """
    LC 474 — Ones and Zeroes
    Approach: 2D 0/1 knapsack — dp[i][j] = max strings with ≤i zeros and ≤j ones.
    Args:
        strs (List[str]): binary strings.
        m (int): maximum allowed zeros.
        n (int): maximum allowed ones.
    Returns:
        int: size of the largest valid subset.
    Time:  O(|strs| * m * n) — each string updates the m×n dp table
    Space: O(m * n) — 2D dp array
    """
    dp = [[0] * (n + 1) for _ in range(m + 1)]  # dp[zeros_used][ones_used]

    for s in strs:
        zeros = s.count('0')   # cost in the first dimension
        ones  = s.count('1')   # cost in the second dimension
        # R→L in both dimensions: 0/1 knapsack (each string at most once)
        for i in range(m, zeros - 1, -1):
            for j in range(n, ones - 1, -1):
                dp[i][j] = max(dp[i][j], dp[i - zeros][j - ones] + 1)

    return dp[m][n]

# Slow motion on strs=["10","0001","111001","1","0"], m=5, n=3:
# s='10': z=1,o=1; update dp[i][j] for i=5..1,j=3..1
#   dp[1][1]=max(0,dp[0][0]+1)=1
# s='0001': z=3,o=1
#   dp[4][2]=max(0,dp[1][1]+1)=2
# s='111001': z=2,o=4 (ones=4>n=3: inner loop empty for j<4)
# s='1': z=0,o=1; dp[i][j]=max(dp[i][j],dp[i][j-1]+1)
# s='0': z=1,o=0; dp[i][j]=max(dp[i][j],dp[i-1][j]+1)
# final dp[5][3]=4

def test_harness(fn):
    tests = [
        (["10","0001","111001","1","0"], 5, 3, 4),
        (["10","0","1"], 1, 1, 2),
        (["0"], 1, 0, 1),
        (["1"], 0, 1, 1),
        (["11","11","0","01"], 2, 3, 3),
    ]
    passed = 0
    for *inputs, expected in tests:
        strs, m, n = inputs
        got = fn(strs, m, n)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | strs={strs} m={m} n={n} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(find_max_form)
print("find_max_form defined.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Coin Change 2 — Count Ways — LC 518

---

```
PROBLEM:
  Given coin denominations and an amount, return the number of combinations
  (not permutations) to make that amount. Infinite supply of each coin.

TRICK:
  UNBOUNDED knapsack count-ways (coins reusable).
  dp[amount] = number of ways to make each amount.
  Key: process coins in the OUTER loop, amounts in INNER loop (LEFT TO RIGHT).
  This gives COMBINATIONS (order doesn't matter).
  If you swapped loops, you'd get PERMUTATIONS.

SLOW MOTION TRACE on amount=5, coins=[1,2,5]:

  dp=[1,0,0,0,0,0]  (1 way to make 0)

  coin=1 (outer loop): for amount 1..5:
    dp=[1,1,1,1,1,1]  (using only 1s)

  coin=2 (outer loop): for amount 2..5:
    dp[2]+=dp[0]=1 → dp[2]=2
    dp[3]+=dp[1]=1 → dp[3]=2
    dp[4]+=dp[2]=2 → dp[4]=3
    dp[5]+=dp[3]=2 → dp[5]=3
    dp=[1,1,2,2,3,3]

  coin=5 (outer loop): for amount 5..5:
    dp[5]+=dp[0]=1 → dp[5]=4
    dp=[1,1,2,2,3,4]

  answer = dp[5] = 4  ({5},{1,1,1,1,1},{1,1,1,2},{1,2,2})

KEY INSIGHT:
  Coin in outer loop, amount in inner loop → combinations (each coin subset counted once).
  Reverse (amount outer, coin inner) → permutations (different orderings counted separately).

TIME:  O(len(coins) * amount)
SPACE: O(amount)
```

In [ ]:
def change(amount, coins):
    """
    LC 518 — Coin Change II
    Approach: Unbounded count-ways knapsack; coin in outer loop for combinations.
    Args:
        amount (int): target amount.
        coins (List[int]): available coin denominations (infinite supply).
    Returns:
        int: number of distinct combinations of coins summing to amount.
    Time:  O(len(coins) * amount) — one pass per coin over the entire dp array
    Space: O(amount) — 1D dp array
    """
    dp = [0] * (amount + 1)
    dp[0] = 1   # 1 way to make amount=0: use no coins

    for coin in coins:          # OUTER loop over coins → combinations, not permutations
        for a in range(coin, amount + 1):   # LEFT TO RIGHT: reuse allowed (unbounded)
            dp[a] += dp[a - coin]           # use this coin once more

    return dp[amount]

# Slow motion on amount=5, coins=[1,2,5]:
# dp=[1,0,0,0,0,0]
# coin=1: dp=[1,1,1,1,1,1]
# coin=2: dp[2]+=dp[0]=1→2; dp[3]+=dp[1]=1→2; dp[4]+=dp[2]=2→3; dp[5]+=dp[3]=2→3
# coin=5: dp[5]+=dp[0]=1→4
# return 4

def test_harness(fn):
    tests = [
        (5, [1,2,5], 4),
        (3, [2], 0),         # impossible
        (10, [10], 1),
        (0, [1,2,3], 1),     # 1 way to make 0
        (500, [1,2,5], 12701),
    ]
    passed = 0
    for *inputs, expected in tests:
        amount, coins = inputs
        got = fn(amount, coins)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | amount={amount} coins={coins} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(change)
print("change defined.")

<a id='9'></a>
## 9. The Knapsack Decision Map

```
QUESTION TYPE                       VARIANT         LOOP           LC
────────────────────────────────────────────────────────────────────────
Partition into 2 equal subsets      0/1 bool        R→L (items)    416
Count ways to assign +/- signs      0/1 count       R→L (items)    494
Max strings with ≤m 0s ≤n 1s        2D 0/1 count    R→L (both dims) 474
Count ways to make change           Unbounded count  L→R (coins out) 518
Min coins to make amount            Unbounded min    L→R (amounts)   322
Can partition into k equal subsets  0/1 bool         backtracking    698
Largest divisible subset            LIS-like DP      L→R             368

THE CRITICAL DISTINCTION:
  COMBINATIONS (order irrelevant): item/coin in OUTER loop
  PERMUTATIONS (order matters):    amount/target in OUTER loop

  Example: coins=[1,2], amount=3
  Combinations: {1,2} counted ONCE → dp=4 (LC 518)
  Permutations: (1,2) and (2,1) both counted → different number

  0/1 (each item once): R→L loop on capacity
  Unbounded (items reusable): L→R loop on capacity
```

<a id='10'></a>
## 10. Interview Cheat Sheet

**1. When to reach for Knapsack DP:**

| Signal | What to Do |
|--------|------------|
| "partition into equal subsets" | 0/1 subset sum, R→L |
| "count ways to assign +/-" | Reduce to count-ways knapsack |
| "two resource constraints" | 2D knapsack, R→L both dims |
| "count combinations to make X" | Unbounded, coin outer, L→R |
| "each item used at most once" | 0/1 knapsack, R→L |
| "items reusable" | Unbounded knapsack, L→R |

**2. The critical direction rule — memorize this:**

```python
# 0/1 KNAPSACK (each item ONCE) — RIGHT TO LEFT
for item_weight in items:
    for cap in range(CAPACITY, item_weight - 1, -1):  # R→L
        dp[cap] = max(dp[cap], dp[cap - item_weight] + item_value)

# UNBOUNDED KNAPSACK (items REUSABLE) — LEFT TO RIGHT
for coin in coins:            # coins outer = combinations
    for a in range(coin, AMOUNT + 1):  # L→R
        dp[a] += dp[a - coin]

# 2D KNAPSACK (two resource limits) — R→L IN BOTH DIMENSIONS
for zeros, ones in [(s.count('0'), s.count('1')) for s in strs]:
    for i in range(m, zeros - 1, -1):
        for j in range(n, ones - 1, -1):
            dp[i][j] = max(dp[i][j], dp[i-zeros][j-ones] + 1)
```

**3. Common templates:**

```python
# TEMPLATE: PARTITION EQUAL SUBSET (LC 416)
total = sum(nums); target = total // 2
if total % 2: return False
dp = [False]*(target+1); dp[0]=True
for num in nums:
    for c in range(target, num-1, -1):
        dp[c] = dp[c] or dp[c-num]
return dp[target]

# TEMPLATE: TARGET SUM REDUCTION (LC 494)
# P + N = total, P - N = target → P = (total+target)/2
if (total+target)%2 or abs(target)>total: return 0
goal = (total+target)//2
dp = [0]*(goal+1); dp[0]=1
for num in nums:
    for c in range(goal, num-1, -1): dp[c]+=dp[c-num]
return dp[goal]
```

**4. Gotchas to not forget:**

```
❌  Going L→R for 0/1 knapsack → items counted multiple times (unbounded behavior)
❌  dp[0]=0 for count-ways → should be dp[0]=1 (1 way to achieve 0: pick nothing)
❌  Not checking (total+target)%2==0 before target sum reduction
❌  Forgetting 2D knapsack iterates BOTH capacity dimensions R→L
✅  COMBINATIONS: coin outer, amount inner, L→R
✅  PERMUTATIONS: amount outer, coin inner, L→R
✅  Subset sum = special case of knapsack where value equals weight
✅  Check early exit: if single num > target in partition, return False immediately
```

## Summary Map

```
                    🎒 KNAPSACK DP
                          │
           ┌──────────────┼──────────────┐
           │              │              │
         0/1           UNBOUNDED       2D
    (each item once)  (items reuse)  (2 limits)
    R→L loop          L→R loop       R→L both
           │              │           LC 474
    ┌──────┴──────┐   ┌───┴───┐
    │             │   │       │
  BOOL         COUNT  COUNT  MAX
  subset sum   ways   ways  value
  dp[c]=or     dp[c]  dp[c]  dp[c]=
  dp[c-w]      +=     +=     max
  LC 416                     LC 518
                   │
             TARGET SUM
             reduce to
             (total+target)/2
             LC 494

CORE RULE:
  0/1: R→L (freeze past, each item once)
  Unbounded: L→R (allow same item again)
  Combinations: item outer, amount inner
  Permutations: amount outer, item inner
```

---
*End of Knapsack DP Master Guide — Sean Edition*